# Week 21: Apache Airflow Fundamentals on MWAA - Authoring and Orchestrating the Fraud Classifier Pipeline

## Where we are

You shipped the fraud classifier in Week 19. You wired up monitoring in Week 20. Last Tuesday the 3am pager went off because a manual notebook run failed. Today you replace those manual notebooks with Airflow DAGs that YOU author in a Databricks cell, upload to MWAA, and trigger from boto3. Every DAG you write today calls the same `fraud-classifier-endpoint` that is already serving traffic in SageMaker.

## Learning objectives

By the end of this 2-hour session you will be able to:

1. AUTHOR a DAG as a Python string in your Databricks notebook, UPLOAD it to the MWAA DAGs bucket via `s3.put_object`, POLL MWAA via `mwaa.invoke_rest_api` until the DAG is registered, then TRIGGER it and read its task logs in the Airflow UI.
2. Wire a multi-step PythonOperator chain that exercises the live `fraud-classifier-endpoint`: read transactions, score them, write predictions, all using XCom to pass S3 URIs (never DataFrames) between tasks.
3. Use `BranchPythonOperator` to route a fraud-scoring DAG to either an alert path or a normal-logging path based on the observed fraud rate, and use the right join-task trigger rule (`none_failed_min_one_success`) so the join does not get silently skipped.
4. Use `ShortCircuitOperator` with `ignore_downstream_trigger_rules=False` to gate a model-approval step on a CloudWatch accuracy metric, AND guarantee a cleanup task with `trigger_rule="all_done"` still runs on a no-go.
5. Wire `on_failure_callback=send_sns_notification(...)` and a fan-in notification task so a failure anywhere in a parallel-scored DAG pages a human via SNS.
6. Recognize when to swap the raw `boto3 sm_runtime.invoke_endpoint` PythonOperator pattern for the bundled `SageMakerTransformOperator` from `apache-airflow-providers-amazon==9.0.0`, and explain why one is the right teaching tool and the other is the right production tool.

## Important: every DAG today is YOURS

Every DAG you trigger today, you also AUTHOR and UPLOAD in this notebook. The only pre-deployed DAG is the smoke DAG you will use as a sanity check. This is the foundation week; Week 22 closes the drift -> retrain -> redeploy loop on top of the DAG-authoring muscle you build today.


## Mental model: Databricks authors, MWAA executes

You are NOT running an Airflow scheduler in this notebook. You never `pip install apache-airflow`. The notebook is the AUTHOR + REMOTE-CONTROL surface; MWAA is the execution surface.

```
+-------------------+        +---------------------+        +-----------------+
|   Databricks      |  put   |   S3 DAGs bucket    |  sync  |     MWAA        |
|   (this notebook) | -----> | bread-academy-      | -----> | scheduler +     |
|                   | object | airflow-dags/dags/  | every  | parser +        |
|   - write DAG as  |        | student_NN/labN.py  |  30s   | workers         |
|     Python string |        +---------------------+        +-----------------+
|   - upload via    |                                              |
|     s3.put_object |                                              | each task calls
|   - trigger via   |   invoke_rest_api    +----------------+      v
|     mwaa.invoke_  | -------------------> | Airflow REST   |   +-----------------+
|     rest_api      |   POST /dagRuns      | API on MWAA    |-->| SageMaker       |
|   - poll for      |                      +----------------+   | fraud-classif-  |
|     status        |                                           | ier-endpoint    |
+-------------------+                                           +-----------------+
                                                                       |
                                                              writes predictions
                                                                       v
                                                              +-----------------+
                                                              | s3://bread-     |
                                                              | academy-shared/ |
                                                              +-----------------+
```

The contract:

- **DAGs live ONLY in MWAA.** The Databricks notebook never imports `airflow`. The DAG source is a Python string until it lands in S3.
- **MWAA syncs the DAGs S3 prefix every 30 seconds**, then the scheduler needs one more parse cycle. Expect 30 to 60 seconds total before a brand-new DAG appears in the API. The poll helper handles that wait.
- **Per-student `dag_id`** of `week21_lab{N}_{STUDENT_ID}` is mandatory. Two students using the same `dag_id` would silently overwrite each other in the scheduler's mind every 30 seconds. There would be no error - the tree view would just oscillate.
- **XCom passes S3 URIs, not data.** XCom values live in the Airflow metadata DB. Push small things (URIs, summary numbers). Use S3 as the data plane.

## The four-step authoring pattern you will use five times today

```
1. dag_code = f'''<DAG source as a Python string with STUDENT_ID baked in>'''
2. s3.put_object(Bucket=DAGS_BUCKET, Key=f"{DAG_PREFIX}/labN.py", Body=dag_code.encode())
3. poll_until_registered(f"week21_labN_{STUDENT_ID}")   # GET /dags/{id} until 200
4. trigger_dag(f"week21_labN_{STUDENT_ID}", conf={...}) # POST /dags/{id}/dagRuns
```

Every lab is a variation on this pattern with one new Airflow concept layered on top.


In [ ]:
# Standard Week 21 setup on Databricks: install the small client surface,
# pull secrets, build boto3 clients. SAME shape as Weeks 19 and 20 so the
# muscle memory carries over. We deliberately do NOT install apache-airflow
# or the Amazon provider here - those run on MWAA, not on Databricks.

%pip install \
    "numpy<2" "pandas<2" \
    "boto3>=1.36" \
    "sagemaker>=2.230,<3" \
    "mlflow-skinny>=2.13,<3" \
    "requests>=2.31"

dbutils.library.restartPython()


In [ ]:
# Pull per-student AWS keys from the secret scope and build boto3 clients.
# Same secret-scope shape as Weeks 19-20: per-student credentials live in
# aws-course-creds-NN, class-wide config in aws-course-shared. No getpass,
# no sagemaker.Session(), no get_execution_role().

import os
import json
import time
import boto3

# Derive identity from the Databricks user. student-NN@... -> that student;
# anyone else (instructor-axel@..., your own login) -> the INSTRUCTOR identity,
# which uses its own creds scope, S3 folder, registry group, and dag_id suffix
# so instructor demo runs never collide with student-01.
_user = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().userName().get()
)
# BREAD_SMOKE_STUDENT_ID forces a specific student number (used by the smoke
# runner). Empty in real runs.
_forced = os.environ.get("BREAD_SMOKE_STUDENT_ID")
IS_INSTRUCTOR = (not _forced) and (not _user.startswith("student-"))

if IS_INSTRUCTOR:
    _num         = "01"                       # instructor uses the cohort-1 endpoint
    USER_SLUG    = "instructor"
    creds_scope  = "aws-course-creds-instructor"
    PACKAGE_GROUP = "fraud-classifier-week19-instructor"
    _folder      = "instructor"
else:
    _num         = _forced or _user.split("@")[0].split("-")[1]
    USER_SLUG    = f"student_{_num}"
    creds_scope  = f"aws-course-creds-{_num}"
    PACKAGE_GROUP = f"fraud-classifier-week19-student-{int(_num):02d}"
    _folder      = f"student-{_num}"

AWS_ACCESS_KEY_ID     = dbutils.secrets.get(scope=creds_scope, key="aws-access-key-id")
AWS_SECRET_ACCESS_KEY = dbutils.secrets.get(scope=creds_scope, key="aws-secret-access-key")
AWS_REGION            = dbutils.secrets.get(scope="aws-course-shared", key="aws-region")

os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_REGION"] = AWS_REGION
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION

# Clients we will use across the notebook.
mwaa             = boto3.client("mwaa", region_name=AWS_REGION)
s3               = boto3.client("s3", region_name=AWS_REGION)
sm_runtime       = boto3.client("sagemaker-runtime", region_name=AWS_REGION)
sagemaker_client = boto3.client("sagemaker", region_name=AWS_REGION)
cloudwatch       = boto3.client("cloudwatch", region_name=AWS_REGION)
sns              = boto3.client("sns", region_name=AWS_REGION)

# Constants. USER_SLUG / PACKAGE_GROUP / _folder were set above per identity.
MWAA_ENV       = "bread-academy-airflow"
# 3 cohorts of 20: students 1-20 -> cohort 1, 21-40 -> 2, 41-60 -> 3. Each
# cohort has its OWN endpoint fraud-classifier-endpoint-cohort-{1,2,3}.
# (The instructor identity maps to cohort 1.)
cohort = (int(_num) - 1) // 20 + 1
ENDPOINT_NAME_BASE = dbutils.secrets.get(scope="aws-course-shared", key="endpoint-name-base")
ENDPOINT_NAME  = f"{ENDPOINT_NAME_BASE}-{cohort}"
SHARED_BUCKET  = "bread-academy-shared"
DAGS_BUCKET    = "bread-academy-airflow-dags"

STUDENT_ID     = _num                       # number used inside baked DAGs / S3 keys
STUDENT_PREFIX = f"week21/{_folder}"        # student-NN or instructor
DAG_PREFIX     = f"dags/{_folder}"

# SNS topic for Labs 2 and 4.
SNS_TOPIC_ARN  = dbutils.secrets.get(scope="aws-course-shared", key="sns-alerts-topic-arn")

print(f"Setup ok: identity={USER_SLUG}, cohort={cohort}, region={AWS_REGION}")
print(f"DAG upload prefix: s3://{DAGS_BUCKET}/{DAG_PREFIX}/")


In [ ]:
# Pre-flight probes - fail loud now if anything we depend on is missing.
# Each probe is one tiny call; the error message tells you who to ask for help.

# (a) MWAA environment is AVAILABLE.
try:
    env = mwaa.get_environment(Name=MWAA_ENV)["Environment"]
    assert env["Status"] == "AVAILABLE", f"MWAA env status is {env['Status']}"
    print(f"MWAA env: AVAILABLE (Airflow {env.get('AirflowVersion')})")
except Exception as e:
    print(f"Ask your instructor to check MWAA environment {MWAA_ENV}.")
    raise

# (b) SageMaker fraud-classifier endpoint is InService.
try:
    resp = sagemaker_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
    assert resp["EndpointStatus"] == "InService", (
        f"Endpoint {ENDPOINT_NAME} is {resp['EndpointStatus']}."
    )
    print(f"Endpoint {ENDPOINT_NAME}: InService")
except Exception as e:
    print(f"Ask your instructor to redeploy the Week 19 endpoint {ENDPOINT_NAME}.")
    raise

# (c) DAGs bucket exists and we can list our prefix.
try:
    s3.head_bucket(Bucket=DAGS_BUCKET)
    print(f"DAGs bucket {DAGS_BUCKET}: reachable")
except Exception as e:
    print(f"Ask your instructor to grant write access on s3://{DAGS_BUCKET}/{DAG_PREFIX}/.")
    raise

# (d) Model package group exists (Lab 3 needs it).
try:
    _pkgs = sagemaker_client.list_model_packages(
        ModelPackageGroupName=PACKAGE_GROUP, MaxResults=1
    )["ModelPackageSummaryList"]
    assert _pkgs, f"per-student group {PACKAGE_GROUP} has no packages"
    print(f"Model package group {PACKAGE_GROUP}: reachable, {len(_pkgs)}+ package(s)")
except Exception as e:
    print(f"Ask your instructor to verify the Week 19 model package group {PACKAGE_GROUP}.")
    raise

# (e) CloudWatch custom metric exists (Lab 3 reads it).
try:
    cloudwatch.list_metrics(Namespace="FraudClassifier", MetricName="Accuracy")
    print("CloudWatch metric FraudClassifier/Accuracy: reachable")
except Exception as e:
    print("Ask your instructor to verify the Week 20 CloudWatch metric.")
    raise

print()
print("Pre-flight checks passed. You are ready to author DAGs.")


## The canonical authoring pattern

Every lab today follows the same four steps. Learn them once; reuse five times.

### Step 1: write the DAG source as a Python string

```python
dag_code = f'''
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime

STUDENT_ID = "{STUDENT_ID}"

def hello(**context):
    print(f"hello from student {{STUDENT_ID}}")

with DAG(
    dag_id=f"week21_lab1_{USER_SLUG}",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
) as dag:
    PythonOperator(task_id="hello", python_callable=hello)
'''
```

The DAG source is an f-string at NOTEBOOK time. `{STUDENT_ID}` is interpolated in the notebook BEFORE upload, so by the time MWAA parses the file it sees a concrete dag_id like `week21_lab1_07`. Any literal `{` or `}` in the DAG source (Python dicts, Jinja templates like `{{ ts_nodash }}`) MUST be double-braced because the OUTER string is itself an f-string.

### Step 2: upload to S3 with `put_object`

```python
s3.put_object(
    Bucket=DAGS_BUCKET,
    Key=f"{DAG_PREFIX}/lab1.py",
    Body=dag_code.encode(),
)
```

Small file. No multipart. `put_object` is the right call.

### Step 3: poll until MWAA has registered the DAG

```python
poll_until_registered(f"week21_lab1_{USER_SLUG}")
```

MWAA syncs the S3 prefix every 30 seconds, then the scheduler needs one more parse cycle. Allow 30 to 60 seconds. The helper loops `GET /dags/{dag_id}` every 5 seconds, up to 20 iterations.

### Step 4: trigger via the Airflow REST API

```python
run_id = trigger_dag(f"week21_lab1_{USER_SLUG}", conf={"batch_size": 50})
wait_for_dag(f"week21_lab1_{USER_SLUG}", run_id)
```

Both calls go through `mwaa.invoke_rest_api`, which hits the same Airflow REST API you would use from a CI pipeline.

### The footgun every student hits at least once

If `poll_until_registered` exits without seeing the DAG, your DAG probably has an IMPORT ERROR. Import errors do NOT bubble back to the boto3 call. They show up in the MWAA Airflow UI:

> Browse -> DAG Import Errors

Open that page first. The error text is usually a one-liner: a stray brace from f-string nesting, a typo, a missing comma.


In [ ]:
# The four helpers that implement the author-upload-poll-trigger pattern.
# Every lab today reuses these verbatim. Read them once now so the labs
# read as "build the dag_code string, call the four helpers".

import time


def upload_dag(student_id: str, lab_n: int, dag_code: str) -> str:
    """Write dag_code to s3://DAGS_BUCKET/dags/student_NN/labN.py and return the key."""
    key = f"dags/student_{student_id}/lab{lab_n}.py"
    s3.put_object(Bucket=DAGS_BUCKET, Key=key, Body=dag_code.encode())
    print(f"Uploaded DAG to s3://{DAGS_BUCKET}/{key} ({len(dag_code)} bytes)")
    return key


def poll_until_registered(dag_id: str, max_iters: int = 40, sleep_s: int = 5) -> None:
    """Block until GET /dags/{dag_id} returns 200, or raise after max_iters * sleep_s seconds.

    MWAA syncs S3 every 30s + 1 parse cycle, so 100s is a safe budget. If we time
    out, the DAG most likely has an import error - check the Airflow UI's
    Browse -> DAG Import Errors panel.
    """
    for i in range(max_iters):
        resp = mwaa.invoke_rest_api(Name=MWAA_ENV, Method="GET", Path=f"/dags/{dag_id}")
        status = resp["RestApiStatusCode"]
        if status == 200:
            print(f"DAG {dag_id} registered after ~{i * sleep_s}s")
            return
        time.sleep(sleep_s)
    raise RuntimeError(
        f"DAG {dag_id} not registered in {max_iters * sleep_s}s. "
        "Open the MWAA Airflow UI and check Browse -> DAG Import Errors."
    )


def unpause_dag(dag_id: str) -> None:
    """Unpause a DAG via PATCH /dags/{dag_id}. MWAA creates new DAGs PAUSED, so
    a trigger does nothing until the DAG is unpaused. Idempotent - safe to call
    every time."""
    resp = mwaa.invoke_rest_api(
        Name=MWAA_ENV,
        Method="PATCH",
        Path=f"/dags/{dag_id}",
        Body={"is_paused": False},
    )
    if resp["RestApiStatusCode"] not in (200,):
        raise RuntimeError(f"unpause_dag failed for {dag_id}: {resp}")
    print(f"Unpaused {dag_id}")


def trigger_dag(dag_id: str, conf: dict | None = None) -> str:
    """POST /dags/{dag_id}/dagRuns with a unique dag_run_id; return the run id."""
    run_id = f"student-{STUDENT_ID}-{int(time.time())}"
    unpause_dag(dag_id)  # MWAA creates DAGs paused; unpause before triggering
    body = {"dag_run_id": run_id}
    if conf is not None:
        body["conf"] = conf
    resp = mwaa.invoke_rest_api(
        Name=MWAA_ENV,
        Method="POST",
        Path=f"/dags/{dag_id}/dagRuns",
        Body=body,
    )
    if resp["RestApiStatusCode"] not in (200, 201):
        raise RuntimeError(f"trigger_dag failed: {resp}")
    print(f"Triggered {dag_id} run_id={run_id}")
    return run_id


def wait_for_dag(dag_id: str, run_id: str, cap_s: int = 900, sleep_s: int = 10) -> str:
    """Poll the run state every sleep_s until it is success or failed, or cap_s elapses."""
    waited = 0
    while waited < cap_s:
        resp = mwaa.invoke_rest_api(
            Name=MWAA_ENV,
            Method="GET",
            Path=f"/dags/{dag_id}/dagRuns/{run_id}",
        )
        state = resp["RestApiResponse"].get("state")
        if state in ("success", "failed"):
            print(f"Run {run_id} finished: {state} (after ~{waited}s)")
            return state
        time.sleep(sleep_s)
        waited += sleep_s
    raise RuntimeError(f"Run {run_id} did not finish in {cap_s}s; still running.")


print("Helpers defined: upload_dag, poll_until_registered, trigger_dag, wait_for_dag")


In [ ]:
# Sanity check: trigger the instructor-deployed smoke DAG and wait for it.
# This is the ONLY DAG today that you did not author yourself - it lives at
# scripts/smoke/dags/week21_smoke_dag.py and is already deployed by your
# instructor. If this works, your four helpers are wired correctly and you
# are ready to start authoring.
#
# The smoke DAG contract: pass `input_s3_key` in dag_run.conf pointing at a
# CSV the DAG can read. The DAG counts rows + fraud-rate and writes a
# _SUCCESS marker back to S3. We synthesize a tiny CSV here so we exercise
# the full round-trip (S3 write -> MWAA reads -> S3 marker -> we read it).

import time

smoke_dag_id = "week21_smoke_dag"

# 1. Upload a tiny synthetic CSV so the DAG has something to read.
synthetic_csv = (
    "transaction_id,amount,is_fraud\n"
    "t1,12.50,0\n"
    "t2,5400.00,1\n"
    "t3,42.10,0\n"
    "t4,89.00,0\n"
)
input_s3_key = f"airflow-smoke/input/student-{STUDENT_ID}-{int(time.time())}.csv"
s3.put_object(
    Bucket=SHARED_BUCKET,
    Key=input_s3_key,
    Body=synthetic_csv.encode("utf-8"),
)
print(f"Uploaded synthetic CSV to s3://{SHARED_BUCKET}/{input_s3_key}")

# 2. Trigger the smoke DAG with the conf shape it expects.
# (No DAG upload, no DAG-discovery poll - the smoke DAG is already registered.)
smoke_run_id = trigger_dag(smoke_dag_id, conf={"input_s3_key": input_s3_key})
smoke_state  = wait_for_dag(smoke_dag_id, smoke_run_id, cap_s=300)

assert smoke_state == "success", f"Smoke DAG failed (state={smoke_state})"

# 3. The smoke DAG writes a marker to S3 - confirm we can read it back.
marker_key = f"airflow-smoke/markers/{smoke_run_id}/_SUCCESS"
try:
    s3.head_object(Bucket=SHARED_BUCKET, Key=marker_key)
    print(f"Smoke marker present: s3://{SHARED_BUCKET}/{marker_key}")
except Exception:
    print(f"Smoke marker NOT found at s3://{SHARED_BUCKET}/{marker_key}")
    print("Run succeeded but marker is missing - tell your instructor.")

print()
print("Helpers + MWAA + S3 round-trip all working. Time to author your own DAGs.")


## Topic 1: PythonOperator, `>>`, and the XCom S3-URI rule

A DAG is a Python file at parse time and a graph of TaskInstances at run time. The minimum-viable shape:

```python
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime

def step_one(**context):
    return "some small value"  # auto-pushed to XCom under key 'return_value'

def step_two(**context):
    ti = context["ti"]
    upstream = ti.xcom_pull(task_ids="step_one")
    print(f"got: {upstream}")

with DAG(
    dag_id="example",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
) as dag:
    one = PythonOperator(task_id="step_one", python_callable=step_one)
    two = PythonOperator(task_id="step_two", python_callable=step_two)
    one >> two   # set dependency: step_two runs AFTER step_one
```

Two rules to internalize before Lab 1:

### Rule 1: PythonOperator callables receive `**context`

```python
def my_task(**context):
    conf      = context["dag_run"].conf      # dict you passed to trigger_dag
    ti        = context["ti"]                # TaskInstance, used for xcom_pull/push
    logical   = context["logical_date"]      # the run's logical timestamp
```

You do not have to declare these arguments explicitly - Airflow injects them.

### Rule 2: XCom carries S3 URIs, not DataFrames

XCom values live in the Airflow metadata DB (Postgres on MWAA). The rule of thumb is "anything bigger than ~1 MB is too big". A pandas DataFrame of 50 predictions might fit; a DataFrame of 50 million rows certainly will not. Use S3 as the data plane, XCom as the control plane:

```python
def pull_batch(**context):
    rows = read_csv_from_s3("s3://...")
    slice_uri = write_csv_to_s3(rows[:50])
    return slice_uri   # one short string in XCom

def score_batch(**context):
    ti = context["ti"]
    slice_uri = ti.xcom_pull(task_ids="pull_batch")
    rows = read_csv_from_s3(slice_uri)
    predictions = score_each_row(rows)
    out_uri = write_jsonl_to_s3(predictions)
    return out_uri
```

The `@dag` decorator exists and is fine. We use `with DAG(...) as dag:` today because it puts the DAG-construction call in the foreground and matches the Airflow documentation samples 1:1.


In [ ]:
# Demo: the EXACT Python source the Lab 1 DAG should produce after you
# f-string-interpolate STUDENT_ID. Read this once, then write your own
# version of it in the Lab 1 starter cell below.
#
# Two tasks:
#   pull_batch  - read 500 rows from s3://bread-academy-shared/lab-inputs/fraud_sample_500.csv,
#                 write a 50-row slice to per-student S3, return that key.
#   score_batch - xcom_pull the slice URI, call sm_runtime.invoke_endpoint per row,
#                 write predictions JSONL back to S3, return the predictions URI.
#
# Inside dag_code the OUTER triple-quoted string is an f-string. Any literal
# Python brace (dicts, Jinja templates like {{ ts_nodash }}) is DOUBLED.

demo_lab1_dag_code = f'''
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime
import boto3, csv, io, json

STUDENT_ID    = "{STUDENT_ID}"
USER_SLUG     = "{USER_SLUG}"
ENDPOINT_NAME = "{ENDPOINT_NAME}"
SHARED_BUCKET = "{SHARED_BUCKET}"
REGION        = "{AWS_REGION}"

INPUT_KEY     = "lab-inputs/fraud_sample_500.csv"
SLICE_KEY     = f"week21/student-{{STUDENT_ID}}/lab1/slice.csv"
PRED_KEY      = f"week21/student-{{STUDENT_ID}}/lab1/predictions.jsonl"


def pull_batch(**context):
    s3 = boto3.client("s3", region_name=REGION)
    body = s3.get_object(Bucket=SHARED_BUCKET, Key=INPUT_KEY)["Body"].read().decode()
    reader = csv.DictReader(io.StringIO(body))
    rows = list(reader)[:50]
    out = io.StringIO()
    writer = csv.DictWriter(out, fieldnames=reader.fieldnames)
    writer.writeheader()
    writer.writerows(rows)
    s3.put_object(Bucket=SHARED_BUCKET, Key=SLICE_KEY, Body=out.getvalue().encode())
    return f"s3://{{SHARED_BUCKET}}/{{SLICE_KEY}}"


def score_batch(**context):
    ti = context["ti"]
    slice_uri = ti.xcom_pull(task_ids="pull_batch")
    bucket, _, key = slice_uri.replace("s3://", "").partition("/")
    s3 = boto3.client("s3", region_name=REGION)
    body = s3.get_object(Bucket=bucket, Key=key)["Body"].read().decode()
    reader = csv.DictReader(io.StringIO(body))
    sm = boto3.client("sagemaker-runtime", region_name=REGION)
    preds = []
    for row in reader:
        resp = sm.invoke_endpoint(
            EndpointName=ENDPOINT_NAME,
            ContentType="application/json",
            Body=json.dumps({{"inputs": row["narrative"]}}).encode(),
        )
        out = json.loads(resp["Body"].read().decode())
        preds.append({{"narrative": row["narrative"], "prediction": out}})
    s3.put_object(
        Bucket=SHARED_BUCKET,
        Key=PRED_KEY,
        Body=("\\n".join(json.dumps(p) for p in preds)).encode(),
    )
    return f"s3://{{SHARED_BUCKET}}/{{PRED_KEY}}"


with DAG(
    dag_id=f"week21_lab1_{USER_SLUG}",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
) as dag:
    t_pull  = PythonOperator(task_id="pull_batch",  python_callable=pull_batch)
    t_score = PythonOperator(task_id="score_batch", python_callable=score_batch)
    t_pull >> t_score
'''

print(demo_lab1_dag_code[:500])
print("... (truncated) ...")
print(f"Total length: {len(demo_lab1_dag_code)} bytes")


In [ ]:
# Demo push (Lab 1): watch the WHOLE pipeline work before you build your own.
# We take the demo source above, rename its dag_id to week21_demo1_<you>
# (a SEPARATE dag from the week21_lab1_<you> you will author in the lab),
# then upload -> unpause -> trigger -> wait. Open the Airflow UI to watch it.
demo1_dag_id = f"week21_demo1_{USER_SLUG}"

# Reuse the demo source but point it at the demo dag_id (so demo and lab stay
# separate in MWAA). The demo source baked dag_id=week21_lab1_<slug>; swap it.
_demo1_src = demo_lab1_dag_code.replace(
    f"week21_lab1_{USER_SLUG}", f"week21_demo1_{USER_SLUG}"
)
# Isolate demo OUTPUT so it never lands in a student's lab prefix: the demo
# source writes under week21/student-<id>/lab1/; point it at week21/<folder>/demo1/.
_demo1_src = _demo1_src.replace(
    "week21/student-{STUDENT_ID}/lab1/", f"week21/{_folder}/demo1/"
)

# Upload under a demo key, register, unpause, trigger, wait.
s3.put_object(
    Bucket=DAGS_BUCKET,
    Key=f"dags/student_{STUDENT_ID}/demo1.py",
    Body=_demo1_src.encode(),
)
print(f"Uploaded demo to dags/student_{STUDENT_ID}/demo1.py")
poll_until_registered(demo1_dag_id)
demo1_run = trigger_dag(demo1_dag_id)   # trigger_dag unpauses first
demo1_state = wait_for_dag(demo1_dag_id, demo1_run)
print(f"Demo 1 finished: {demo1_state} (dag_id={demo1_dag_id})")
print("Open the Airflow UI and find this dag to see the graph + task logs.")
print("Now go build YOUR version in the lab below.")


## Lab 1: Author and upload your first DAG (~15 min)

Your turn. You will author, upload, and trigger a two-task DAG that pulls 50 fraud-transaction rows from S3 and scores them through the live `fraud-classifier-endpoint`.

### Goal

Build a Python f-string `lab1_dag_code` whose `dag_id` is `week21_lab1_{USER_SLUG}` (with your student id baked in). The DAG has two PythonOperator tasks - `pull_batch` and `score_batch` - chained by `>>`. Upload via `upload_dag`, poll via `poll_until_registered`, trigger via `trigger_dag`, wait via `wait_for_dag`. Then read the predictions JSONL from S3 and compute the fraud rate.

### Steps

1. Build the f-string. Look at the demo cell above and reproduce its shape. Bake your `STUDENT_ID` into the source.
2. `upload_dag(STUDENT_ID, 1, lab1_dag_code)`.
3. `poll_until_registered(f"week21_lab1_{USER_SLUG}")`. Expect 30 to 60 seconds.
4. `lab1_run_id = trigger_dag(f"week21_lab1_{USER_SLUG}")`.
5. `wait_for_dag(...)` and assert the final state is `"success"`.
6. Read the predictions JSONL from `s3://bread-academy-shared/week21/student-NN/lab1/predictions.jsonl`. Compute `lab1_fraud_rate` as the fraction of rows whose prediction label is `"fraud"`.
7. Optionally also pull the `score_batch` task's XCom return value via the REST API:

   ```
   GET /dags/{dag_id}/dagRuns/{run_id}/taskInstances/score_batch/xcomEntries/return_value
   ```

### Final assertions

- The predictions JSONL has exactly 50 lines.
- `0.0 <= lab1_fraud_rate <= 1.0`.

### Stretch (in class, if you have time)

Trigger the Lab 1 DAG a second time. Notice that re-running `upload_dag` overwrites the same `.py` file in S3 and the dag_id stays the same; the scheduler picks up the new file but does NOT create a new dag. This is the right mental model for "deploy a new version of the DAG".

### Homework extension

Extend `score_batch` to also write a Parquet copy of the predictions alongside the JSONL. Use `pyarrow` (already on the MWAA workers). Re-upload and re-trigger. What does this teach you about the cost of changing a DAG vs the cost of changing a downstream consumer?


In [ ]:
# Lab 1 starter (scaffolded). The DAG skeleton is given. You fill the TWO
# task BODIES marked YOUR CODE - you do NOT need to retype the whole file.
# Do NOT run the safety-net cell below if you finish this on your own.
#
# Reminder: dag_code is an f-string, so STUDENT_ID etc. are baked at upload
# time. Any LITERAL brace inside (dicts, Jinja) must be DOUBLED: {{ }}.

lab1_dag_code = f"""
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime
import boto3, csv, io, json

STUDENT_ID    = "{STUDENT_ID}"
USER_SLUG     = "{USER_SLUG}"
ENDPOINT_NAME = "{ENDPOINT_NAME}"
SHARED_BUCKET = "{SHARED_BUCKET}"
REGION        = "{AWS_REGION}"

INPUT_KEY = "lab-inputs/fraud_sample_500.csv"
SLICE_KEY = f"week21/student-{{STUDENT_ID}}/lab1/slice.csv"
PRED_KEY  = f"week21/student-{{STUDENT_ID}}/lab1/predictions.jsonl"


def pull_batch(**context):
    # YOUR CODE
    # Read INPUT_KEY from SHARED_BUCKET, keep the first 50 rows, write them
    # to SLICE_KEY, and return f"s3://{{SHARED_BUCKET}}/{{SLICE_KEY}}".
    pass


def score_batch(**context):
    # YOUR CODE
    # xcom_pull the slice URI from pull_batch, read the CSV, call
    # sm_runtime.invoke_endpoint on each row's narrative, write the
    # predictions to PRED_KEY as JSONL, return its s3:// URI.
    pass


with DAG(
    dag_id=f"week21_lab1_{USER_SLUG}",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
) as dag:
    t_pull  = PythonOperator(task_id="pull_batch",  python_callable=pull_batch)
    t_score = PythonOperator(task_id="score_batch", python_callable=score_batch)
    t_pull >> t_score
"""

# Upload, poll, trigger, wait. Then collect the run id and predictions URI.
lab1_run_id = None  # YOUR CODE
lab1_predictions_uri = None  # YOUR CODE

# Read the predictions JSONL and compute the fraud rate.
lab1_fraud_rate = None  # YOUR CODE

# Final assertions (uncomment when ready):
# assert lab1_dag_code is not None and "week21_lab1_" in lab1_dag_code
# assert lab1_run_id is not None
# assert lab1_predictions_uri and lab1_predictions_uri.startswith("s3://")
# assert 0.0 <= lab1_fraud_rate <= 1.0
# print(f"Lab 1 PASS: fraud_rate={lab1_fraud_rate:.3f}")


In [ ]:
# Lab 1 SAFETY-NET: run this if you didn't finish Lab 1 so the rest of the
# notebook still works. SKIP this cell if you DID finish Lab 1.

if lab1_run_id is None:
    print("Using Lab 1 safety-net so the rest of the notebook can run.")
    lab1_dag_code = demo_lab1_dag_code   # the demo above is the reference solution
    upload_dag(STUDENT_ID, 1, lab1_dag_code)
    poll_until_registered(f"week21_lab1_{USER_SLUG}")
    lab1_run_id = trigger_dag(f"week21_lab1_{USER_SLUG}")
    state = wait_for_dag(f"week21_lab1_{USER_SLUG}", lab1_run_id)
    assert state == "success", f"Lab 1 DAG ended in state {state}"

    lab1_predictions_uri = f"s3://{SHARED_BUCKET}/week21/student-{STUDENT_ID}/lab1/predictions.jsonl"
    body = s3.get_object(
        Bucket=SHARED_BUCKET,
        Key=f"week21/student-{STUDENT_ID}/lab1/predictions.jsonl",
    )["Body"].read().decode()
    lines = [json.loads(line) for line in body.strip().splitlines()]

    def _is_fraud(p):
        pred = p["prediction"]
        if isinstance(pred, list) and pred:
            pred = pred[0]
        return isinstance(pred, dict) and pred.get("label") == "fraud"

    lab1_fraud_rate = sum(1 for p in lines if _is_fraud(p)) / max(len(lines), 1)
    print(f"Safety-net Lab 1 PASS: predictions={len(lines)}, fraud_rate={lab1_fraud_rate:.3f}")


## Think About It

Two short reflections - jot down your answers; we will not collect them.

1. **XCom carries S3 URIs, not DataFrames.** Why is passing the S3 URI through XCom safer than passing the predictions DataFrame directly? Imagine a downstream team also wants the predictions for an audit pipeline next quarter - which design is easier to extend?

2. **dag_id collisions are silent.** Your DAG is `week21_lab1_{USER_SLUG}`. What would the Airflow scheduler do if two students used the literal `week21_lab1` without the suffix and both uploaded to the same prefix? Hint: there is no error log. The Airflow GitHub issue tracker calls this "silently overrides DAGs with duplicate dag_id" - the tree view oscillates between definitions every 30 seconds. Why is the per-student suffix the right answer for a 60-student class?


## Topic 2: BranchPythonOperator and the join trigger-rule footgun

A DAG often has to make a runtime decision: "if fraud rate is high, page someone; otherwise, just log it." `BranchPythonOperator`'s callable returns a `task_id` (or list of task_ids); the chosen branch arms run, and the UNCHOSEN arms become `skipped`.

```
                  pull_batch
                     |
                 score_batch
                     |
              compute_fraud_rate
                     |
                decide_route        <- BranchPythonOperator
                /         \
        high_risk          normal
              \           /
                 join             <- the footgun lives here
```

### The footgun

The default trigger rule on every task is `all_success`. That rule fires only when ALL upstream tasks succeeded. After a branch, at least one upstream is `skipped` (not `success`), so a join task with the default trigger rule is ALSO skipped. Silently. The DAG looks green but `join` never ran.

### The fix

Set the join task's `trigger_rule` to `none_failed_min_one_success`:

```python
from airflow.utils.trigger_rule import TriggerRule

join = EmptyOperator(
    task_id="join",
    trigger_rule=TriggerRule.NONE_FAILED_MIN_ONE_SUCCESS,
)
```

This means: fire if no upstream FAILED AND at least one upstream succeeded. A `skipped` upstream is fine. This is the canonical fix documented in Airflow GitHub issue #19222.

### Lab 2 shape

`pull_batch >> score_batch >> compute_fraud_rate >> decide_route` branches to either `high_risk` (writes a marker to `s3://bread-academy-shared/high-risk/...` AND publishes SNS) or `normal` (just logs). Both arms then converge on `join`, an `EmptyOperator` with the trigger rule above. Run it twice - once with a low threshold so `high_risk` fires, once with a high threshold so `normal` fires.


In [ ]:
# Demo: the Lab 2 DAG. Same pull/score pattern as Lab 1, plus a fraud-rate
# computation, a BranchPythonOperator, two branch arms, and a join with the
# all-important none_failed_min_one_success trigger rule.
#
# The threshold lives in dag_run.conf so we can flip the branch decision
# from the trigger call without re-uploading the DAG.

demo_lab2_dag_code = f'''
from airflow import DAG
from airflow.operators.python import PythonOperator, BranchPythonOperator
from airflow.operators.empty import EmptyOperator
from airflow.utils.trigger_rule import TriggerRule
from datetime import datetime
import boto3, csv, io, json

STUDENT_ID    = "{STUDENT_ID}"
USER_SLUG     = "{USER_SLUG}"
ENDPOINT_NAME = "{ENDPOINT_NAME}"
SHARED_BUCKET = "{SHARED_BUCKET}"
REGION        = "{AWS_REGION}"
SNS_TOPIC_ARN = "{SNS_TOPIC_ARN}"

INPUT_KEY  = "lab-inputs/fraud_sample_500.csv"
SLICE_KEY  = f"week21/student-{{STUDENT_ID}}/lab2/slice.csv"
PRED_KEY   = f"week21/student-{{STUDENT_ID}}/lab2/predictions.jsonl"
HIGH_RISK_KEY = f"high-risk/student-{{STUDENT_ID}}/marker.json"


def pull_batch(**context):
    s3 = boto3.client("s3", region_name=REGION)
    body = s3.get_object(Bucket=SHARED_BUCKET, Key=INPUT_KEY)["Body"].read().decode()
    reader = csv.DictReader(io.StringIO(body))
    rows = list(reader)[:50]
    out = io.StringIO()
    writer = csv.DictWriter(out, fieldnames=reader.fieldnames)
    writer.writeheader()
    writer.writerows(rows)
    s3.put_object(Bucket=SHARED_BUCKET, Key=SLICE_KEY, Body=out.getvalue().encode())
    return f"s3://{{SHARED_BUCKET}}/{{SLICE_KEY}}"


def score_batch(**context):
    ti = context["ti"]
    slice_uri = ti.xcom_pull(task_ids="pull_batch")
    bucket, _, key = slice_uri.replace("s3://", "").partition("/")
    s3 = boto3.client("s3", region_name=REGION)
    body = s3.get_object(Bucket=bucket, Key=key)["Body"].read().decode()
    reader = csv.DictReader(io.StringIO(body))
    sm = boto3.client("sagemaker-runtime", region_name=REGION)
    preds = []
    for row in reader:
        resp = sm.invoke_endpoint(
            EndpointName=ENDPOINT_NAME,
            ContentType="application/json",
            Body=json.dumps({{"inputs": row["narrative"]}}).encode(),
        )
        out = json.loads(resp["Body"].read().decode())
        preds.append({{"narrative": row["narrative"], "prediction": out}})
    s3.put_object(
        Bucket=SHARED_BUCKET,
        Key=PRED_KEY,
        Body=("\\n".join(json.dumps(p) for p in preds)).encode(),
    )
    return f"s3://{{SHARED_BUCKET}}/{{PRED_KEY}}"


def compute_fraud_rate(**context):
    ti = context["ti"]
    pred_uri = ti.xcom_pull(task_ids="score_batch")
    bucket, _, key = pred_uri.replace("s3://", "").partition("/")
    s3 = boto3.client("s3", region_name=REGION)
    body = s3.get_object(Bucket=bucket, Key=key)["Body"].read().decode()
    lines = [json.loads(line) for line in body.strip().splitlines()]
    def is_fraud(p):
        pred = p["prediction"]
        if isinstance(pred, list) and pred:
            pred = pred[0]
        return isinstance(pred, dict) and pred.get("label") == "fraud"
    rate = sum(1 for p in lines if is_fraud(p)) / max(len(lines), 1)
    return rate


def decide_route(**context):
    ti = context["ti"]
    rate = ti.xcom_pull(task_ids="compute_fraud_rate")
    threshold = context["dag_run"].conf.get("fraud_rate_threshold", 0.05)
    return "high_risk" if rate > threshold else "normal"


def high_risk(**context):
    ti = context["ti"]
    rate = ti.xcom_pull(task_ids="compute_fraud_rate")
    s3 = boto3.client("s3", region_name=REGION)
    s3.put_object(
        Bucket=SHARED_BUCKET,
        Key=HIGH_RISK_KEY,
        Body=json.dumps({{"fraud_rate": rate, "student_id": STUDENT_ID}}).encode(),
    )
    sns = boto3.client("sns", region_name=REGION)
    sns.publish(
        TopicArn=SNS_TOPIC_ARN,
        Subject="High fraud rate detected",
        Message=f"student {{STUDENT_ID}} fraud_rate={{rate:.3f}}",
    )
    print(f"HIGH RISK: rate={{rate:.3f}}")


def normal(**context):
    ti = context["ti"]
    rate = ti.xcom_pull(task_ids="compute_fraud_rate")
    print(f"NORMAL: rate={{rate:.3f}}")


with DAG(
    dag_id=f"week21_lab2_{USER_SLUG}",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
) as dag:
    t_pull   = PythonOperator(task_id="pull_batch",  python_callable=pull_batch)
    t_score  = PythonOperator(task_id="score_batch", python_callable=score_batch)
    t_rate   = PythonOperator(task_id="compute_fraud_rate", python_callable=compute_fraud_rate)
    t_decide = BranchPythonOperator(task_id="decide_route", python_callable=decide_route)
    t_high   = PythonOperator(task_id="high_risk", python_callable=high_risk)
    t_norm   = PythonOperator(task_id="normal",    python_callable=normal)
    # Remove the trigger_rule below to see the join silently skip on every run.
    t_join   = EmptyOperator(
        task_id="join",
        trigger_rule=TriggerRule.NONE_FAILED_MIN_ONE_SUCCESS,
    )
    t_pull >> t_score >> t_rate >> t_decide >> [t_high, t_norm] >> t_join
'''

print(f"Lab 2 DAG source length: {len(demo_lab2_dag_code)} bytes")


In [ ]:
# Demo push (Lab 2): watch the WHOLE pipeline work before you build your own.
# We take the demo source above, rename its dag_id to week21_demo2_<you>
# (a SEPARATE dag from the week21_lab2_<you> you will author in the lab),
# then upload -> unpause -> trigger -> wait. Open the Airflow UI to watch it.
demo2_dag_id = f"week21_demo2_{USER_SLUG}"

# Reuse the demo source but point it at the demo dag_id (so demo and lab stay
# separate in MWAA). The demo source baked dag_id=week21_lab2_<slug>; swap it.
_demo2_src = demo_lab2_dag_code.replace(
    f"week21_lab2_{USER_SLUG}", f"week21_demo2_{USER_SLUG}"
)
# Isolate demo OUTPUT so it never lands in a student's lab prefix: the demo
# source writes under week21/student-<id>/lab2/; point it at week21/<folder>/demo2/.
_demo2_src = _demo2_src.replace(
    "week21/student-{STUDENT_ID}/lab2/", f"week21/{_folder}/demo2/"
)

# Upload under a demo key, register, unpause, trigger, wait.
s3.put_object(
    Bucket=DAGS_BUCKET,
    Key=f"dags/student_{STUDENT_ID}/demo2.py",
    Body=_demo2_src.encode(),
)
print(f"Uploaded demo to dags/student_{STUDENT_ID}/demo2.py")
poll_until_registered(demo2_dag_id)
demo2_run = trigger_dag(demo2_dag_id)   # trigger_dag unpauses first
demo2_state = wait_for_dag(demo2_dag_id, demo2_run)
print(f"Demo 2 finished: {demo2_state} (dag_id={demo2_dag_id})")
print("Open the Airflow UI and find this dag to see the graph + task logs.")
print("Now go build YOUR version in the lab below.")


## Lab 2: Author the branching DAG (~15 min)

You will author a DAG that runs the same `pull_batch >> score_batch` chain, then computes the fraud rate, then BRANCHES based on it.

### Goal

Build `lab2_dag_code` whose `dag_id` is `week21_lab2_{USER_SLUG}` and shape is:

```
pull_batch >> score_batch >> compute_fraud_rate >> decide_route -> [high_risk, normal] >> join
```

- `decide_route` is a `BranchPythonOperator` that reads `dag_run.conf["fraud_rate_threshold"]` (default 0.05) and returns `"high_risk"` if the observed rate exceeds it, else `"normal"`.
- `high_risk` writes a marker to `s3://bread-academy-shared/high-risk/student-NN/marker.json` AND publishes SNS.
- `normal` just prints.
- `join` is an `EmptyOperator` with `trigger_rule=TriggerRule.NONE_FAILED_MIN_ONE_SUCCESS`.

### Steps

1. Author the DAG (look at the demo above), upload via `upload_dag(..., 2, ...)`, poll until registered.
2. `lab2_high_risk_run_id = trigger_dag(..., conf={"fraud_rate_threshold": 0.05})`. The pre-loaded input is engineered to have ~7-10% fraud, so `high_risk` should fire. Wait for success.
3. `lab2_normal_run_id = trigger_dag(..., conf={"fraud_rate_threshold": 0.99})`. With this threshold no observed rate can exceed it, so `normal` should fire. Wait for success.
4. Open the MWAA Airflow UI graph view for both runs and confirm visually which branch ran and which is `skipped`. The `join` task should be `success` in BOTH runs.

### Final assertions

- Both runs ended in state `"success"`.
- In the high-risk run, `s3://bread-academy-shared/high-risk/student-NN/marker.json` exists.
- In the normal run, the marker is unchanged from the high-risk run (because the `high_risk` task did not execute in the normal run).

### Stretch (in class)

Edit your `lab2_dag_code` to REMOVE the `trigger_rule=...` line on the `join` task (so it defaults to `all_success`). Re-upload and re-trigger either flavor. Open the UI graph view: `join` will be `skipped` even though the rest of the DAG succeeded. Revert the change before moving on.

### Homework extension

Turn `decide_route` into a 3-way branch (`low`, `medium`, `high`) and write the corresponding `>>` graph. What new trigger rule does the join need? Hint: it does not change - `none_failed_min_one_success` already handles "one of three arms ran".


In [ ]:
# Lab 2 starter. Author the branching DAG, upload it, and trigger it twice
# with different thresholds.

# (1) Author the DAG source. Make sure 'join' has trigger_rule=NONE_FAILED_MIN_ONE_SUCCESS.
lab2_dag_code = None  # YOUR CODE

# (2) Trigger with the low threshold to make high_risk fire.
lab2_high_risk_run_id = None  # YOUR CODE

# (3) Trigger with the high threshold to make normal fire.
lab2_normal_run_id = None  # YOUR CODE

# Final assertions (uncomment when ready):
# assert lab2_dag_code is not None and "BranchPythonOperator" in lab2_dag_code
# assert lab2_high_risk_run_id is not None
# assert lab2_normal_run_id is not None
# print("Lab 2 PASS: both runs completed; check the UI graph view to confirm branches.")


In [ ]:
# Lab 2 SAFETY-NET: run this if you didn't finish Lab 2 so the rest of the
# notebook still works. SKIP this cell if you DID finish Lab 2.

if lab2_high_risk_run_id is None:
    print("Using Lab 2 safety-net so the rest of the notebook can run.")
    lab2_dag_code = demo_lab2_dag_code
    upload_dag(STUDENT_ID, 2, lab2_dag_code)
    poll_until_registered(f"week21_lab2_{USER_SLUG}")

    lab2_high_risk_run_id = trigger_dag(
        f"week21_lab2_{USER_SLUG}", conf={"fraud_rate_threshold": 0.05}
    )
    s1 = wait_for_dag(f"week21_lab2_{USER_SLUG}", lab2_high_risk_run_id)
    assert s1 == "success", f"high-risk run ended in {s1}"

    lab2_normal_run_id = trigger_dag(
        f"week21_lab2_{USER_SLUG}", conf={"fraud_rate_threshold": 0.99}
    )
    s2 = wait_for_dag(f"week21_lab2_{USER_SLUG}", lab2_normal_run_id)
    assert s2 == "success", f"normal run ended in {s2}"

    print("Safety-net Lab 2 PASS: both runs success; high_risk and normal each ran once.")


## Topic 3: ShortCircuit gates and the cleanup-task trigger rule

A common ML-Ops pattern: gate a deployment on a recent quality metric. "Approve the new model package ONLY if the live endpoint's CloudWatch accuracy was above 0.85 over the last hour. Otherwise, skip the approval - but still record what happened."

`ShortCircuitOperator` is built for this:

```python
from airflow.operators.python import ShortCircuitOperator

gate = ShortCircuitOperator(
    task_id="gate_on_accuracy",
    python_callable=should_approve,           # returns True/False
    ignore_downstream_trigger_rules=False,    # <- critical, see below
)
```

When the callable returns truthy, downstream tasks proceed normally. When it returns falsy, downstream tasks are SKIPPED.

### The footgun

`ignore_downstream_trigger_rules` defaults to `True`. With that default, a falsy gate skips EVERY downstream task in the DAG, recursively, regardless of their `trigger_rule`. Your `all_done` cleanup task that you carefully designed to run on success OR failure? Skipped. Silently.

### The fix

Set `ignore_downstream_trigger_rules=False`. Now the skip only propagates to DIRECT downstream tasks; indirect downstreams get to honor their own trigger rules.

### The Lab 3 shape

```
fetch_accuracy_from_cloudwatch
            |
       gate_on_accuracy         <- ShortCircuit, ignore_downstream_trigger_rules=False
            |
   approve_model_package        <- skipped on no-go
            |
       notify_outcome           <- trigger_rule="all_done" - runs even on gate skip
```

`notify_outcome` writes a marker to S3 saying either `"approved"` or `"gated"`. Run the DAG twice - once with a low accuracy threshold (gate passes, both downstream tasks run), once with a very high threshold (gate fails, `approve_model_package` is skipped, `notify_outcome` STILL runs and records `"gated"`).


In [ ]:
# Demo: the Lab 3 DAG. Three tasks - fetch the latest CloudWatch accuracy,
# gate on it, approve the most recent model package - plus an always-running
# notify_outcome cleanup task with trigger_rule="all_done". The whole point
# of this DAG is to demonstrate ignore_downstream_trigger_rules=False working
# correctly: notify_outcome MUST fire in both the approved and gated branches.

demo_lab3_dag_code = f'''
from airflow import DAG
from airflow.operators.python import PythonOperator, ShortCircuitOperator
from airflow.utils.trigger_rule import TriggerRule
from datetime import datetime, timedelta, timezone
import boto3, json

STUDENT_ID    = "{STUDENT_ID}"
USER_SLUG     = "{USER_SLUG}"
REGION        = "{AWS_REGION}"
SHARED_BUCKET = "{SHARED_BUCKET}"
PACKAGE_GROUP = "{PACKAGE_GROUP}"

OUTCOME_KEY = f"week21/student-{{STUDENT_ID}}/lab3/outcome.json"


def fetch_accuracy(**context):
    cw = boto3.client("cloudwatch", region_name=REGION)
    now = datetime.now(timezone.utc)
    resp = cw.get_metric_statistics(
        Namespace="FraudClassifier",
        MetricName="Accuracy",
        StartTime=now - timedelta(hours=1),
        EndTime=now,
        Period=300,
        Statistics=["Average"],
    )
    datapoints = sorted(resp.get("Datapoints", []), key=lambda d: d["Timestamp"])
    if not datapoints:
        # Pre-class fallback so the lab is robust even if the metric is fresh.
        return 0.90
    return float(datapoints[-1]["Average"])


def gate_on_accuracy(**context):
    ti = context["ti"]
    acc = ti.xcom_pull(task_ids="fetch_accuracy")
    threshold = context["dag_run"].conf.get("min_accuracy", 0.85)
    print(f"accuracy={{acc:.3f}} threshold={{threshold:.3f}}")
    return acc > threshold


def approve_model_package(**context):
    sm = boto3.client("sagemaker", region_name=REGION)
    pkgs = sm.list_model_packages(
        ModelPackageGroupName=PACKAGE_GROUP,
        MaxResults=1,
        SortBy="CreationTime",
        SortOrder="Descending",
    )["ModelPackageSummaryList"]
    if not pkgs:
        raise RuntimeError(f"No packages in group {{PACKAGE_GROUP}}")
    arn = pkgs[0]["ModelPackageArn"]
    sm.update_model_package(
        ModelPackageArn=arn,
        ModelApprovalStatus="Approved",
        ApprovalDescription=f"Approved by week21_lab3_{{USER_SLUG}}",
    )
    print(f"Approved {{arn}}")


def notify_outcome(**context):
    ti = context["ti"]
    approved_state = ti.xcom_pull(task_ids="approve_model_package", default=None)
    gate_result = ti.xcom_pull(task_ids="gate_on_accuracy")
    outcome = "approved" if gate_result else "gated"
    s3 = boto3.client("s3", region_name=REGION)
    s3.put_object(
        Bucket=SHARED_BUCKET,
        Key=OUTCOME_KEY,
        Body=json.dumps({{"outcome": outcome, "student_id": STUDENT_ID}}).encode(),
    )
    print(f"notify_outcome wrote outcome={{outcome}}")


with DAG(
    dag_id=f"week21_lab3_{USER_SLUG}",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
) as dag:
    t_fetch  = PythonOperator(task_id="fetch_accuracy",  python_callable=fetch_accuracy)
    t_gate   = ShortCircuitOperator(
        task_id="gate_on_accuracy",
        python_callable=gate_on_accuracy,
        ignore_downstream_trigger_rules=False,   # the whole point
    )
    t_approve = PythonOperator(task_id="approve_model_package", python_callable=approve_model_package)
    t_notify  = PythonOperator(
        task_id="notify_outcome",
        python_callable=notify_outcome,
        trigger_rule=TriggerRule.ALL_DONE,
    )
    t_fetch >> t_gate >> t_approve >> t_notify
    t_gate >> t_notify   # also wire gate directly to notify so the all_done rule has an upstream even on skip
'''

print(f"Lab 3 DAG source length: {len(demo_lab3_dag_code)} bytes")


In [ ]:
# Demo push (Lab 3): watch the WHOLE pipeline work before you build your own.
# We take the demo source above, rename its dag_id to week21_demo3_<you>
# (a SEPARATE dag from the week21_lab3_<you> you will author in the lab),
# then upload -> unpause -> trigger -> wait. Open the Airflow UI to watch it.
demo3_dag_id = f"week21_demo3_{USER_SLUG}"

# Reuse the demo source but point it at the demo dag_id (so demo and lab stay
# separate in MWAA). The demo source baked dag_id=week21_lab3_<slug>; swap it.
_demo3_src = demo_lab3_dag_code.replace(
    f"week21_lab3_{USER_SLUG}", f"week21_demo3_{USER_SLUG}"
)
# Isolate demo OUTPUT so it never lands in a student's lab prefix: the demo
# source writes under week21/student-<id>/lab3/; point it at week21/<folder>/demo3/.
_demo3_src = _demo3_src.replace(
    "week21/student-{STUDENT_ID}/lab3/", f"week21/{_folder}/demo3/"
)

# Upload under a demo key, register, unpause, trigger, wait.
s3.put_object(
    Bucket=DAGS_BUCKET,
    Key=f"dags/student_{STUDENT_ID}/demo3.py",
    Body=_demo3_src.encode(),
)
print(f"Uploaded demo to dags/student_{STUDENT_ID}/demo3.py")
poll_until_registered(demo3_dag_id)
demo3_run = trigger_dag(demo3_dag_id)   # trigger_dag unpauses first
demo3_state = wait_for_dag(demo3_dag_id, demo3_run)
print(f"Demo 3 finished: {demo3_state} (dag_id={demo3_dag_id})")
print("Open the Airflow UI and find this dag to see the graph + task logs.")
print("Now go build YOUR version in the lab below.")


## Lab 3: Author the accuracy gate DAG (~15 min)

You will author a DAG that fetches the live CloudWatch accuracy metric, gates a model-package approval on it, and ALWAYS records the outcome via a cleanup task.

### Goal

Build `lab3_dag_code` whose `dag_id` is `week21_lab3_{USER_SLUG}` and shape is:

```
fetch_accuracy >> gate_on_accuracy >> approve_model_package >> notify_outcome
                                |__________________________________^
```

- `gate_on_accuracy` is a `ShortCircuitOperator` with `ignore_downstream_trigger_rules=False`. Its callable returns `True` if the fetched accuracy is above `dag_run.conf["min_accuracy"]`.
- `approve_model_package` calls `sagemaker.update_model_package` on the most recent package in your per-student group `fraud-classifier-week19-student-NN` with `ModelApprovalStatus="Approved"`.
- `notify_outcome` has `trigger_rule="all_done"` and writes a JSON marker to `s3://bread-academy-shared/week21/student-NN/lab3/outcome.json` containing `{"outcome": "approved" or "gated"}`.

### Steps

1. Author the DAG, upload via `upload_dag(..., 3, ...)`, poll until registered.
2. **Pass run**: `trigger_dag(..., conf={"min_accuracy": 0.5})`. Wait for success. Read the outcome marker; it should be `"approved"`.
3. **Gated run**: `trigger_dag(..., conf={"min_accuracy": 0.99})`. Wait for success (the DAG itself does NOT fail just because the gate skipped downstream). Read the outcome marker; it should be `"gated"`.

### Final assertions

- `lab3_pass_outcome == "approved"`.
- `lab3_gated_outcome == "gated"`.

### Stretch (in class)

Re-author the DAG with `ignore_downstream_trigger_rules=True` (the default), re-upload, re-trigger the gated run. Look at the UI: `notify_outcome` is `skipped` even though it has `trigger_rule="all_done"`. THIS is the silent footgun that just lost you the cleanup write.

### Homework extension

Replace `ShortCircuitOperator` with a `BranchPythonOperator` whose callable returns `"approve"`, `"reject"`, or `"pending"` based on the accuracy value, where each option leads to a different downstream task. Now you have a 3-state gate with a proper audit trail (different markers per outcome).


In [ ]:
# Lab 3 starter. Author the gate DAG and trigger it twice with different
# accuracy thresholds.

# (1) Author the DAG source. Remember ignore_downstream_trigger_rules=False on the gate.
lab3_dag_code = None  # YOUR CODE

# (2) Pass run (low threshold).
lab3_pass_run = None  # YOUR CODE

# (3) Gated run (high threshold).
lab3_gated_run = None  # YOUR CODE

# (4) Read both outcome markers from S3.
lab3_pass_outcome  = None  # YOUR CODE
lab3_gated_outcome = None  # YOUR CODE

# Final assertions (uncomment when ready):
# assert lab3_pass_outcome == "approved"
# assert lab3_gated_outcome == "gated"
# print("Lab 3 PASS: gate respected ignore_downstream_trigger_rules=False")


In [ ]:
# Lab 3 SAFETY-NET: run this if you didn't finish Lab 3 so the rest of the
# notebook still works. SKIP this cell if you DID finish Lab 3.

if lab3_pass_run is None:
    print("Using Lab 3 safety-net so the rest of the notebook can run.")
    lab3_dag_code = demo_lab3_dag_code
    upload_dag(STUDENT_ID, 3, lab3_dag_code)
    poll_until_registered(f"week21_lab3_{USER_SLUG}")

    lab3_pass_run = trigger_dag(
        f"week21_lab3_{USER_SLUG}", conf={"min_accuracy": 0.5}
    )
    sp = wait_for_dag(f"week21_lab3_{USER_SLUG}", lab3_pass_run)
    assert sp == "success", f"pass run ended in {sp}"
    body = s3.get_object(
        Bucket=SHARED_BUCKET,
        Key=f"week21/student-{STUDENT_ID}/lab3/outcome.json",
    )["Body"].read().decode()
    lab3_pass_outcome = json.loads(body)["outcome"]

    lab3_gated_run = trigger_dag(
        f"week21_lab3_{USER_SLUG}", conf={"min_accuracy": 0.99}
    )
    sg = wait_for_dag(f"week21_lab3_{USER_SLUG}", lab3_gated_run)
    assert sg == "success", f"gated run ended in {sg}"
    body = s3.get_object(
        Bucket=SHARED_BUCKET,
        Key=f"week21/student-{STUDENT_ID}/lab3/outcome.json",
    )["Body"].read().decode()
    lab3_gated_outcome = json.loads(body)["outcome"]

    print(f"Safety-net Lab 3 PASS: pass_outcome={lab3_pass_outcome}, gated_outcome={lab3_gated_outcome}")


## Think About It

1. **Gating on the right signal.** Your gate is on a CloudWatch accuracy metric, which captures live model quality. What other signals could a Bread Financial team gate on? Examples: false-positive rate over the last day, chargeback rate from the operations team, a downstream feature-store freshness metric. What is the operational cost of each signal - how often does it update, who maintains it, how long does staleness lie undetected?

2. **The cost of a silent cleanup skip.** Why is `ignore_downstream_trigger_rules=False` mandatory when you have an `all_done` cleanup task? Imagine your cleanup writes an audit row to a compliance table. What does an "I forgot to set that flag" outage look like three months later when a regulator asks for the audit trail?


## Topic 4: Parallelism, fan-in, and `on_failure_callback`

Airflow lets you express fan-out + fan-in in one line:

```
pull_batch >> [score_a, score_b, score_c] >> notify_done
```

The three `score_*` tasks run in parallel (up to your worker slot cap; MWAA `mw1.small` is 5 concurrent slots, so 3 parallel scorers is well within budget). `notify_done` fan-ins them.

### Two trigger-rule choices for the fan-in

- `trigger_rule="all_success"` (the default): `notify_done` only fires if all three scorers succeeded.
- `trigger_rule="none_failed"`: `notify_done` fires as long as no scorer outright failed. Skipped scorers are tolerated.

For Lab 4 we want `notify_done` to fire on a fully-green run but NOT on a run where one partition failed (we want the SNS failure callback to be the only signal in that case). So `none_failed` is correct.

### Page a human on ANY task failure

`default_args` propagates to every task:

```python
from airflow.providers.amazon.aws.notifications.sns import send_sns_notification

notifier = send_sns_notification(
    aws_conn_id="aws_default",
    region_name="us-west-2",
    target_arn=SNS_TOPIC_ARN,
    message="Task {{ ti.task_id }} in {{ dag.dag_id }} failed at {{ ts }}",
)

default_args = {
    "on_failure_callback": [notifier],   # MUST be a list in provider 9.0.0
}
```

Now any task that ends in `failed` triggers an SNS publish via Airflow's callback machinery. The Jinja-templated message includes the task id, dag id, and timestamp.

### The Lab 4 shape

```
pull_batch >> [score_partition_a, score_partition_b, score_partition_c] >> notify_done
```

`score_partition_b` deliberately raises `RuntimeError` when `dag_run.conf["force_fail"] is True`. Trigger the DAG once with `force_fail=False` (everyone succeeds, notify_done publishes the "all complete" SNS) and once with `force_fail=True` (partition B fails, `on_failure_callback` fires the failure SNS, `notify_done` is skipped because of `trigger_rule="none_failed"`).


In [ ]:
# Demo: the Lab 4 DAG. Fan-out into three partition scorers, fan-in into
# notify_done. on_failure_callback at the DAG level fires SNS on any task
# failure. Partition B has an injectable failure via dag_run.conf["force_fail"].

demo_lab4_dag_code = f'''
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.providers.amazon.aws.notifications.sns import send_sns_notification
from airflow.utils.trigger_rule import TriggerRule
from datetime import datetime
import boto3, csv, io, json

STUDENT_ID    = "{STUDENT_ID}"
USER_SLUG     = "{USER_SLUG}"
ENDPOINT_NAME = "{ENDPOINT_NAME}"
SHARED_BUCKET = "{SHARED_BUCKET}"
REGION        = "{AWS_REGION}"
SNS_TOPIC_ARN = "{SNS_TOPIC_ARN}"

INPUT_KEY = "lab-inputs/fraud_sample_500.csv"
SLICE_KEY = f"week21/student-{{STUDENT_ID}}/lab4/slice.csv"


def pull_batch(**context):
    s3 = boto3.client("s3", region_name=REGION)
    body = s3.get_object(Bucket=SHARED_BUCKET, Key=INPUT_KEY)["Body"].read().decode()
    reader = csv.DictReader(io.StringIO(body))
    rows = list(reader)[:90]
    out = io.StringIO()
    writer = csv.DictWriter(out, fieldnames=reader.fieldnames)
    writer.writeheader()
    writer.writerows(rows)
    s3.put_object(Bucket=SHARED_BUCKET, Key=SLICE_KEY, Body=out.getvalue().encode())
    return f"s3://{{SHARED_BUCKET}}/{{SLICE_KEY}}"


def _score_partition(start, end, partition_name, **context):
    ti = context["ti"]
    slice_uri = ti.xcom_pull(task_ids="pull_batch")
    bucket, _, key = slice_uri.replace("s3://", "").partition("/")
    s3 = boto3.client("s3", region_name=REGION)
    body = s3.get_object(Bucket=bucket, Key=key)["Body"].read().decode()
    reader = list(csv.DictReader(io.StringIO(body)))[start:end]
    sm = boto3.client("sagemaker-runtime", region_name=REGION)
    preds = []
    for row in reader:
        resp = sm.invoke_endpoint(
            EndpointName=ENDPOINT_NAME,
            ContentType="application/json",
            Body=json.dumps({{"inputs": row["narrative"]}}).encode(),
        )
        out = json.loads(resp["Body"].read().decode())
        preds.append({{"narrative": row["narrative"], "prediction": out}})
    out_key = f"week21/student-{{STUDENT_ID}}/lab4/{{partition_name}}.jsonl"
    s3.put_object(
        Bucket=SHARED_BUCKET,
        Key=out_key,
        Body=("\\n".join(json.dumps(p) for p in preds)).encode(),
    )
    return f"s3://{{SHARED_BUCKET}}/{{out_key}}"


def score_partition_a(**context):
    return _score_partition(0, 30, "a", **context)


def score_partition_b(**context):
    if context["dag_run"].conf.get("force_fail", False):
        raise RuntimeError("forced failure in partition B for the alert demo")
    return _score_partition(30, 60, "b", **context)


def score_partition_c(**context):
    return _score_partition(60, 90, "c", **context)


def notify_done(**context):
    sns = boto3.client("sns", region_name=REGION)
    sns.publish(
        TopicArn=SNS_TOPIC_ARN,
        Subject="Lab 4 fan-in complete",
        Message=f"student {{STUDENT_ID}}: all 3 partitions scored",
    )
    print("notify_done published SNS")


sns_failure = send_sns_notification(
    aws_conn_id="aws_default",
    region_name=REGION,
    target_arn=SNS_TOPIC_ARN,
    message="Task {{{{ ti.task_id }}}} in {{{{ dag.dag_id }}}} failed at {{{{ ts }}}}",
)

default_args = {{
    "on_failure_callback": [sns_failure],
}}


with DAG(
    dag_id=f"week21_lab4_{USER_SLUG}",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
    default_args=default_args,
) as dag:
    t_pull = PythonOperator(task_id="pull_batch", python_callable=pull_batch)
    t_a = PythonOperator(task_id="score_partition_a", python_callable=score_partition_a)
    t_b = PythonOperator(task_id="score_partition_b", python_callable=score_partition_b)
    t_c = PythonOperator(task_id="score_partition_c", python_callable=score_partition_c)
    t_notify = PythonOperator(
        task_id="notify_done",
        python_callable=notify_done,
        trigger_rule=TriggerRule.NONE_FAILED,
    )
    t_pull >> [t_a, t_b, t_c] >> t_notify
'''

print(f"Lab 4 DAG source length: {len(demo_lab4_dag_code)} bytes")


In [ ]:
# Demo push (Lab 4): watch the WHOLE pipeline work before you build your own.
# We take the demo source above, rename its dag_id to week21_demo4_<you>
# (a SEPARATE dag from the week21_lab4_<you> you will author in the lab),
# then upload -> unpause -> trigger -> wait. Open the Airflow UI to watch it.
demo4_dag_id = f"week21_demo4_{USER_SLUG}"

# Reuse the demo source but point it at the demo dag_id (so demo and lab stay
# separate in MWAA). The demo source baked dag_id=week21_lab4_<slug>; swap it.
_demo4_src = demo_lab4_dag_code.replace(
    f"week21_lab4_{USER_SLUG}", f"week21_demo4_{USER_SLUG}"
)
# Isolate demo OUTPUT so it never lands in a student's lab prefix: the demo
# source writes under week21/student-<id>/lab4/; point it at week21/<folder>/demo4/.
_demo4_src = _demo4_src.replace(
    "week21/student-{STUDENT_ID}/lab4/", f"week21/{_folder}/demo4/"
)

# Upload under a demo key, register, unpause, trigger, wait.
s3.put_object(
    Bucket=DAGS_BUCKET,
    Key=f"dags/student_{STUDENT_ID}/demo4.py",
    Body=_demo4_src.encode(),
)
print(f"Uploaded demo to dags/student_{STUDENT_ID}/demo4.py")
poll_until_registered(demo4_dag_id)
demo4_run = trigger_dag(demo4_dag_id)   # trigger_dag unpauses first
demo4_state = wait_for_dag(demo4_dag_id, demo4_run)
print(f"Demo 4 finished: {demo4_state} (dag_id={demo4_dag_id})")
print("Open the Airflow UI and find this dag to see the graph + task logs.")
print("Now go build YOUR version in the lab below.")


## Lab 4: Author the fan-in alerting DAG (~15 min)

You will author a DAG that scores transactions in three parallel partitions, fans in to a notification task, and pages a human via SNS on ANY task failure.

### Goal

Build `lab4_dag_code` whose `dag_id` is `week21_lab4_{USER_SLUG}` and shape is:

```
pull_batch >> [score_partition_a, score_partition_b, score_partition_c] >> notify_done
```

- `default_args={"on_failure_callback": [send_sns_notification(...)]}` so every task wires the failure SNS automatically.
- `notify_done` has `trigger_rule="none_failed"`.
- `score_partition_b` raises `RuntimeError` when `dag_run.conf["force_fail"]` is True.

### Steps

1. Make sure your email is subscribed to the SNS topic (your instructor may have done this for the class - ask if you are unsure).
2. Author the DAG, upload via `upload_dag(..., 4, ...)`, poll until registered.
3. **OK run**: `trigger_dag(..., conf={"force_fail": False})`. Wait for success. You should receive a "Lab 4 fan-in complete" email.
4. **Fail run**: `trigger_dag(..., conf={"force_fail": True})`. Wait for the final state, which should be `"failed"`. You should receive a "Task score_partition_b in week21_lab4_NN failed at ..." email.

### Final assertions

- `lab4_ok_run` ended in `"success"`.
- `lab4_fail_run` ended in `"failed"`.

### Stretch (in class)

Change `notify_done`'s `trigger_rule` to `"all_done"` and re-upload. Re-trigger the fail run. Now `notify_done` also runs after the partition failure - and publishes the success SNS. This is exactly the alert-storm risk we are trying to avoid. Revert.

### Homework extension

Add `on_failure_callback=[send_sns_notification(...)]` to your Lab 1 DAG. Trigger it once normally and once with a deliberate failure (e.g. make `score_batch` call an endpoint that does not exist). What alert-storm risks do you create if every DAG in your team publishes to the same SNS topic? How would you triage which alerts mean "wake up at 3am" vs "look at it tomorrow"?


In [ ]:
# Lab 4 starter. Author the fan-in DAG with on_failure_callback at the
# default_args level, then trigger it twice (clean and force_fail).

# (1) Author the DAG source.
lab4_dag_code = None  # YOUR CODE

# (2) Clean run.
lab4_ok_run = None  # YOUR CODE

# (3) Force-fail run.
lab4_fail_run = None  # YOUR CODE

# Final assertions (uncomment when ready):
# assert lab4_dag_code is not None and "on_failure_callback" in lab4_dag_code
# assert lab4_ok_run is not None
# assert lab4_fail_run is not None
# print("Lab 4 PASS: check your inbox for two SNS emails (success and failure).")


In [ ]:
# Lab 4 SAFETY-NET: run this if you didn't finish Lab 4 so the rest of the
# notebook still works. SKIP this cell if you DID finish Lab 4.

if lab4_ok_run is None:
    print("Using Lab 4 safety-net so the rest of the notebook can run.")
    lab4_dag_code = demo_lab4_dag_code
    upload_dag(STUDENT_ID, 4, lab4_dag_code)
    poll_until_registered(f"week21_lab4_{USER_SLUG}")

    lab4_ok_run = trigger_dag(
        f"week21_lab4_{USER_SLUG}", conf={"force_fail": False}
    )
    sok = wait_for_dag(f"week21_lab4_{USER_SLUG}", lab4_ok_run)
    print(f"ok run state: {sok}")

    lab4_fail_run = trigger_dag(
        f"week21_lab4_{USER_SLUG}", conf={"force_fail": True}
    )
    sfail = wait_for_dag(f"week21_lab4_{USER_SLUG}", lab4_fail_run)
    print(f"fail run state: {sfail}")

    print("Safety-net Lab 4 PASS: check your inbox for two SNS emails (success and failure).")


## Topic 5: When to reach for the SageMaker provider operator

Labs 1 through 4 all scored transactions by calling `boto3 sm_runtime.invoke_endpoint` inside a `PythonOperator`. That is the right tool for REALTIME, per-record inference - and it is the only choice because `apache-airflow-providers-amazon` does NOT ship a realtime-invoke operator.

For BATCH inference, the provider DOES ship a purpose-built operator:

### The provider 9.0.0 SageMaker operator catalog (highlights)

- `SageMakerTrainingOperator` - run a training job.
- `SageMakerProcessingOperator` - run a processing job (feature engineering, evaluation).
- `SageMakerTransformOperator` - run a batch transform (read S3 input, score, write S3 output).
- `SageMakerEndpointOperator` - CREATE or UPDATE an endpoint. **Does NOT invoke endpoints.**
- `SageMakerRegisterModelVersionOperator` - register a new model package version (Week 22 will lean on this).

### The rule of thumb

| Pattern | Operator |
|---|---|
| Realtime, per-record inference | `PythonOperator` + `boto3 sm_runtime.invoke_endpoint` |
| Batch inference on a file in S3 | `SageMakerTransformOperator` |
| Training a model | `SageMakerTrainingOperator` |
| Promoting a model version | `SageMakerRegisterModelVersionOperator` |

### Why this matters

`SageMakerTransformOperator` handles the create-transform-job + wait-for-completion + check-status mechanics for you. With `PythonOperator + boto3` you would write those three calls by hand (and likely forget the wait loop). The provider operator is the same amount of code in your DAG but trades "your bug surface" for "Airflow community's tested code".

In Lab 5 you will rewrite Lab 1 using `SageMakerTransformOperator` against a batch input file.


In [ ]:
# Demo: the Lab 5 DAG. Three tasks - build the transform input, run the
# SageMakerTransformOperator, summarize the output. Notice we never call
# sm_runtime.invoke_endpoint - the operator handles create_transform_job +
# wait_for_completion behind the scenes. Import path is:
#   from airflow.providers.amazon.aws.operators.sagemaker import SageMakerTransformOperator
#
# ModelName here ("fraud-classifier-week19-model") is the SageMaker Model
# resource your instructor created from the Week 19 model package. The
# Transform operator needs a Model resource, NOT an endpoint name.

demo_lab5_dag_code = f'''
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.providers.amazon.aws.operators.sagemaker import SageMakerTransformOperator
from datetime import datetime
import boto3, csv, io, json

STUDENT_ID    = "{STUDENT_ID}"
USER_SLUG     = "{USER_SLUG}"
SHARED_BUCKET = "{SHARED_BUCKET}"
REGION        = "{AWS_REGION}"
MODEL_NAME    = "fraud-classifier-week19-model"

INPUT_KEY  = "lab-inputs/fraud_sample_500.csv"
LAB5_INPUT_KEY  = f"week21/student-{{STUDENT_ID}}/lab5/input/input.csv"
LAB5_OUTPUT_PREFIX = f"week21/student-{{STUDENT_ID}}/lab5/output/"


def build_transform_input(**context):
    s3 = boto3.client("s3", region_name=REGION)
    body = s3.get_object(Bucket=SHARED_BUCKET, Key=INPUT_KEY)["Body"].read().decode()
    reader = csv.DictReader(io.StringIO(body))
    rows = list(reader)[:50]
    out = io.StringIO()
    writer = csv.writer(out)
    for r in rows:
        writer.writerow([r["narrative"]])
    s3.put_object(Bucket=SHARED_BUCKET, Key=LAB5_INPUT_KEY, Body=out.getvalue().encode())
    return f"s3://{{SHARED_BUCKET}}/{{LAB5_INPUT_KEY}}"


def summarize_results(**context):
    s3 = boto3.client("s3", region_name=REGION)
    objs = s3.list_objects_v2(
        Bucket=SHARED_BUCKET, Prefix=LAB5_OUTPUT_PREFIX
    ).get("Contents", [])
    if not objs:
        raise RuntimeError(f"No transform output under s3://{{SHARED_BUCKET}}/{{LAB5_OUTPUT_PREFIX}}")
    total_rows = 0
    fraud_rows = 0
    for o in objs:
        body = s3.get_object(Bucket=SHARED_BUCKET, Key=o["Key"])["Body"].read().decode()
        for line in body.strip().splitlines():
            total_rows += 1
            try:
                rec = json.loads(line)
                pred = rec[0] if isinstance(rec, list) and rec else rec
                if isinstance(pred, dict) and pred.get("label") == "fraud":
                    fraud_rows += 1
            except Exception:
                pass
    print(f"transform output: {{total_rows}} rows, fraud_rate={{fraud_rows / max(total_rows, 1):.3f}}")


transform_config = {{
    "TransformJobName": f"week21-lab5-{STUDENT_ID}-{{{{ ts_nodash }}}}",
    "ModelName": MODEL_NAME,
    "TransformInput": {{
        "DataSource": {{
            "S3DataSource": {{
                "S3DataType": "S3Prefix",
                "S3Uri": f"s3://{{SHARED_BUCKET}}/week21/student-{{STUDENT_ID}}/lab5/input/",
            }}
        }},
        "ContentType": "text/csv",
        "SplitType": "Line",
    }},
    "TransformOutput": {{
        "S3OutputPath": f"s3://{{SHARED_BUCKET}}/week21/student-{{STUDENT_ID}}/lab5/output/",
        "AssembleWith": "Line",
    }},
    "TransformResources": {{
        "InstanceCount": 1,
        "InstanceType": "ml.m5.large",
    }},
}}


with DAG(
    dag_id=f"week21_lab5_{USER_SLUG}",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
) as dag:
    t_build   = PythonOperator(task_id="build_transform_input", python_callable=build_transform_input)
    t_transform = SageMakerTransformOperator(
        task_id="run_transform",
        config=transform_config,
        wait_for_completion=True,
    )
    t_summarize = PythonOperator(task_id="summarize_results", python_callable=summarize_results)
    t_build >> t_transform >> t_summarize
'''

print(f"Lab 5 DAG source length: {len(demo_lab5_dag_code)} bytes")


In [ ]:
# Demo push (Lab 5): watch the WHOLE pipeline work before you build your own.
# We take the demo source above, rename its dag_id to week21_demo5_<you>
# (a SEPARATE dag from the week21_lab5_<you> you will author in the lab),
# then upload -> unpause -> trigger -> wait. Open the Airflow UI to watch it.
demo5_dag_id = f"week21_demo5_{USER_SLUG}"

# Reuse the demo source but point it at the demo dag_id (so demo and lab stay
# separate in MWAA). The demo source baked dag_id=week21_lab5_<slug>; swap it.
_demo5_src = demo_lab5_dag_code.replace(
    f"week21_lab5_{USER_SLUG}", f"week21_demo5_{USER_SLUG}"
)
# Isolate demo OUTPUT so it never lands in a student's lab prefix: the demo
# source writes under week21/student-<id>/lab5/; point it at week21/<folder>/demo5/.
_demo5_src = _demo5_src.replace(
    "week21/student-{STUDENT_ID}/lab5/", f"week21/{_folder}/demo5/"
)

# Upload under a demo key, register, unpause, trigger, wait.
s3.put_object(
    Bucket=DAGS_BUCKET,
    Key=f"dags/student_{STUDENT_ID}/demo5.py",
    Body=_demo5_src.encode(),
)
print(f"Uploaded demo to dags/student_{STUDENT_ID}/demo5.py")
poll_until_registered(demo5_dag_id)
demo5_run = trigger_dag(demo5_dag_id)   # trigger_dag unpauses first
demo5_state = wait_for_dag(demo5_dag_id, demo5_run)
print(f"Demo 5 finished: {demo5_state} (dag_id={demo5_dag_id})")
print("Open the Airflow UI and find this dag to see the graph + task logs.")
print("Now go build YOUR version in the lab below.")


## Lab 5: Rewrite Lab 1 with the provider operator (~10 min)

You will rewrite Lab 1 using `SageMakerTransformOperator` instead of `PythonOperator + boto3 sm_runtime.invoke_endpoint`. The DAG goes from "realtime per-record" to "batch over an S3 file" - the right shape when you are scoring a whole CSV.

### Goal

Build `lab5_dag_code` whose `dag_id` is `week21_lab5_{USER_SLUG}` and shape is:

```
build_transform_input >> run_transform >> summarize_results
```

- `build_transform_input` writes a one-column CSV of `narrative` text to `s3://bread-academy-shared/week21/student-NN/lab5/input/input.csv`.
- `run_transform` is a `SageMakerTransformOperator` with `ModelName="fraud-classifier-week19-model"`, S3 input/output, `InstanceType="ml.m5.large"`, `wait_for_completion=True`.
- `summarize_results` reads the output JSONL from `s3://bread-academy-shared/week21/student-NN/lab5/output/` and prints the fraud rate.

### Steps

1. Author the DAG, upload via `upload_dag(..., 5, ...)`, poll until registered.
2. `lab5_run_id = trigger_dag(...)`. Wait for success (the transform job takes ~3-5 minutes; bump `cap_s` in `wait_for_dag` to 1200).
3. List the output prefix in S3 and set `lab5_output_uri` to one of the result keys.

### Final assertions

- `lab5_run_id` ended in `"success"`.
- `lab5_output_uri` exists in S3 (use `s3.head_object` to verify).

### Stretch (in class)

Change `InstanceType` to `ml.m5.xlarge` in the config. Re-upload, re-trigger. Watch the Airflow task log - the transform finishes faster but costs more per minute. This is a real-world cost-vs-latency knob.

### Homework extension

Write a one-paragraph answer (in your notes) to: "When would I reach for `SageMakerTransformOperator` vs `PythonOperator + boto3 sm_runtime.invoke_endpoint`?" Give three specific examples from a Bread Financial context. Hint: think about batch nightly scoring vs realtime decisioning at the point of sale.


In [ ]:
# Lab 5 starter. Rewrite Lab 1 with SageMakerTransformOperator.

# (1) Author the DAG source.
lab5_dag_code = None  # YOUR CODE

# (2) Trigger and wait (transform jobs can take 3-5 min; use cap_s=1200).
lab5_run_id = None  # YOUR CODE

# (3) Confirm an output object exists.
lab5_output_uri = None  # YOUR CODE

# Final assertions (uncomment when ready):
# assert lab5_dag_code is not None and "SageMakerTransformOperator" in lab5_dag_code
# assert lab5_run_id is not None
# assert lab5_output_uri and lab5_output_uri.startswith("s3://")
# print("Lab 5 PASS: provider-operator batch transform completed.")


In [ ]:
# Lab 5 SAFETY-NET: run this if you didn't finish Lab 5 so the rest of the
# notebook still works. SKIP this cell if you DID finish Lab 5.

if lab5_run_id is None:
    print("Using Lab 5 safety-net so the rest of the notebook can run.")
    lab5_dag_code = demo_lab5_dag_code
    upload_dag(STUDENT_ID, 5, lab5_dag_code)
    poll_until_registered(f"week21_lab5_{USER_SLUG}")
    lab5_run_id = trigger_dag(f"week21_lab5_{USER_SLUG}")
    state = wait_for_dag(f"week21_lab5_{USER_SLUG}", lab5_run_id, cap_s=1200)
    assert state == "success", f"Lab 5 DAG ended in state {state}"

    objs = s3.list_objects_v2(
        Bucket=SHARED_BUCKET,
        Prefix=f"week21/student-{STUDENT_ID}/lab5/output/",
    ).get("Contents", [])
    assert objs, "Transform output prefix is empty"
    lab5_output_uri = f"s3://{SHARED_BUCKET}/{objs[0]['Key']}"
    print(f"Safety-net Lab 5 PASS: output_uri={lab5_output_uri}")


## Wrap-up

### What we did today

In one 2-hour session you AUTHORED, UPLOADED, and TRIGGERED five DAGs that exercise the live `fraud-classifier-endpoint`. Every DAG you wrote landed at `s3://bread-academy-airflow-dags/dags/student_{STUDENT_ID}/lab{N}.py` and showed up in the MWAA Airflow UI under a per-student `dag_id`. The author-upload-poll-trigger pattern is now muscle memory.

### What is in your toolbox now

1. The four-step author-upload-poll-trigger pattern with `s3.put_object` + `mwaa.invoke_rest_api`.
2. `PythonOperator` chains with `>>`, and the XCom S3-URI rule.
3. `BranchPythonOperator` plus the `none_failed_min_one_success` join trigger rule that prevents silent join skips.
4. `ShortCircuitOperator` with `ignore_downstream_trigger_rules=False` plus an `all_done` cleanup task.
5. `default_args["on_failure_callback"] = [send_sns_notification(...)]` for paging, plus `trigger_rule="none_failed"` for clean fan-in.
6. `SageMakerTransformOperator` from `apache-airflow-providers-amazon` for batch inference - and the rule of thumb for when to reach for it vs PythonOperator + boto3.

### Homework recap

1. **Lab 1**: Extend `score_batch` to also write a Parquet copy alongside the JSONL.
2. **Lab 2**: Turn `decide_route` into a 3-way branch (`low`, `medium`, `high`).
3. **Lab 3**: Replace the ShortCircuit gate with a `BranchPythonOperator` that has three outcomes (`approve`, `reject`, `pending`).
4. **Lab 4**: Add `on_failure_callback` to your Lab 1 DAG and reflect on alert-storm risks.
5. **Lab 5**: Write a paragraph on when to reach for `SageMakerTransformOperator` vs `PythonOperator + boto3 sm_runtime.invoke_endpoint`. Three specific Bread Financial examples.

### Next week (Week 22)

You will close the loop: a single DAG that detects drift, retrains the model, and redeploys the endpoint - all built on top of the same author-upload-poll-trigger muscle you have today. You will use `SageMakerRegisterModelVersionOperator`, the Datasets / data-aware scheduling primitive (covered in the optional notebook for this week), and the same gate pattern from Lab 3.

### Reference reading

- AWS MWAA "Adding or updating DAGs" - the S3 sync contract.
- Astronomer "Best practices for orchestrating MLOps pipelines with Airflow".
- Apache Airflow stable REST API reference - `/dags/{dag_id}` and `/dags/{dag_id}/dagRuns`.
- `apache-airflow-providers-amazon` 9.0.0 SageMaker operator catalog.
